# Step 12 — Divergence & Momentum Workbench

This notebook explores cross-asset divergence and momentum features
added in D17–D19. Key questions:

1. Do divergence z-scores spike at regime transitions (leading indicator)?
2. Which momentum features track regime changes most closely?
3. Do divergence triggers fire before transitions?
4. Are divergence/momentum features redundant with each other?

**Requires:** pipeline steps 1-3 (features + cluster labels).

## Setup + Load Features (D10.1)

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "../src")
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from trading_crab_lib.config import load, setup_logging
from trading_crab_lib.runtime import RunConfig
from trading_crab_lib.checkpoints import CheckpointManager
from trading_crab_lib import DATA_DIR, OUTPUT_DIR, plotting

setup_logging("INFO")
log = logging.getLogger("12_divergence_momentum")
cfg = load()
run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=False)
cm = CheckpointManager()

In [ ]:
# Load features and labels
features = None
kmeans_labels = None

try:
    features = cm.load("features")
    print(f"Features loaded: {features.shape}")
except Exception as e:
    print(f"Features not found: {e}")
    print("Run: python run_pipeline.py --steps 1,2")

try:
    cluster_labels = cm.load("cluster_labels")
    if "balanced_cluster" in cluster_labels.columns:
        kmeans_labels = cluster_labels["balanced_cluster"]
    elif "cluster" in cluster_labels.columns:
        kmeans_labels = cluster_labels["cluster"]
    print(f"KMeans labels loaded: {kmeans_labels.nunique()} regimes")
except Exception:
    print("Cluster labels not found — run step 3.")

# Identify divergence and momentum columns
div_cols = [c for c in (features.columns if features is not None else [])
            if c.startswith("div_")]
mom_cols = [c for c in (features.columns if features is not None else [])
            if "_mom_" in c or "_rs_" in c or "acceleration" in c or "corr_" in c]
z_cols = [c for c in div_cols if "_z_" in c]
trigger_cols = [c for c in div_cols if "_trigger" in c]

print(f"\nDivergence columns: {len(div_cols)} ({len(z_cols)} z-scores, {len(trigger_cols)} triggers)")
print(f"Momentum columns: {len(mom_cols)}")
if div_cols:
    print(f"  Div: {div_cols[:8]}{'...' if len(div_cols) > 8 else ''}")
if mom_cols:
    print(f"  Mom: {mom_cols[:8]}{'...' if len(mom_cols) > 8 else ''}")

## Divergence Z-Score Time-Series (D10.2)

Z-score divergence features over time with vertical markers at regime
transitions. If divergence spikes precede transitions, the features
are useful leading indicators.

In [ ]:
if features is not None and kmeans_labels is not None and z_cols:
    plotting.plot_divergence_timeseries(
        features, kmeans_labels, run_cfg,
        cols=z_cols,
        filename="12_divergence_timeseries.png",
    )
    print(f"Plotted {len(z_cols)} divergence z-score columns.")
elif not z_cols:
    print("No divergence z-score columns found in features.")
    print("Divergence features are computed in step 2 if source data is available.")
else:
    print("Features or labels not available.")

## Momentum Dashboard (D10.3)

Trailing momentum, relative strength, cross-correlations, and CPI
acceleration — colored by regime. Shows which momentum features
differentiate regimes visually.

In [ ]:
if features is not None and kmeans_labels is not None and mom_cols:
    plotting.plot_momentum_dashboard(
        features, kmeans_labels, run_cfg,
        cols=mom_cols,
        filename="12_momentum_dashboard.png",
    )
    print(f"Plotted {len(mom_cols)} momentum columns.")
elif not mom_cols:
    print("No momentum columns found in features.")
    print("Momentum features are computed in step 2 if source data is available.")
else:
    print("Features or labels not available.")

## Divergence Trigger Analysis (D10.4)

Do divergence triggers fire before regime transitions? For each trigger
column, compute the fraction of regime transitions that were preceded
by a trigger firing in the prior 1–4 quarters. Compare against baseline
(random) trigger frequency.

In [ ]:
if features is not None and kmeans_labels is not None and trigger_cols:
    common = features.index.intersection(kmeans_labels.dropna().index)
    labels = kmeans_labels.loc[common].astype(int)
    feat = features.loc[common]

    # Find transition quarters (where label changes)
    transitions = labels.index[labels.diff().fillna(0) != 0]
    # Remove first index (no prior quarter to compare)
    transitions = transitions[1:]
    n_transitions = len(transitions)
    print(f"Regime transitions detected: {n_transitions}")

    rows = []
    for tcol in trigger_cols:
        if tcol not in feat.columns:
            continue
        trigger = feat[tcol].fillna(0).astype(bool)
        baseline_rate = trigger.mean()

        # Check if trigger fired in lookback window before each transition
        for lookback in [1, 2, 4]:
            preceded_count = 0
            for t in transitions:
                idx_pos = labels.index.get_loc(t)
                start = max(0, idx_pos - lookback)
                window = trigger.iloc[start:idx_pos]
                if window.any():
                    preceded_count += 1

            preceded_pct = preceded_count / n_transitions if n_transitions > 0 else 0
            rows.append({
                "Trigger": tcol,
                "Lookback (Q)": lookback,
                "% Transitions Preceded": f"{preceded_pct:.1%}",
                "Baseline Rate": f"{baseline_rate:.1%}",
                "Lift": f"{preceded_pct / baseline_rate:.1f}x" if baseline_rate > 0 else "N/A",
            })

    if rows:
        result = pd.DataFrame(rows)
        print("\nDivergence Trigger Leading Indicator Analysis:")
        display(result)

        # Bar chart: preceded % vs baseline for 2Q lookback
        two_q = result[result["Lookback (Q)"] == 2].copy()
        if not two_q.empty:
            fig, ax = plt.subplots(figsize=(10, max(3, len(two_q) * 0.6)))
            y_pos = range(len(two_q))
            preceded = [float(v.strip("%")) / 100 for v in two_q["% Transitions Preceded"]]
            baseline = [float(v.strip("%")) / 100 for v in two_q["Baseline Rate"]]
            ax.barh(y_pos, preceded, height=0.35, color=plotting.CUSTOM_COLORS[0],
                    alpha=0.8, label="% Transitions Preceded (2Q)")
            ax.barh([y + 0.35 for y in y_pos], baseline, height=0.35,
                    color=plotting.CUSTOM_COLORS[1], alpha=0.6, label="Baseline Rate")
            ax.set_yticks([y + 0.175 for y in y_pos])
            ax.set_yticklabels(two_q["Trigger"].values, fontsize=8)
            ax.set_xlabel("Rate")
            ax.set_title("Divergence Triggers as Leading Indicators (2Q lookback)", fontsize=11)
            ax.legend(fontsize=8)
            ax.grid(axis="x", alpha=0.2)
            fig.tight_layout()
            plotting._save_or_show(fig, "12_trigger_leading_indicator.png", run_cfg)
    else:
        print("No trigger analysis results.")
elif not trigger_cols:
    print("No divergence trigger columns found in features.")
else:
    print("Features or labels not available.")

## Feature Correlation — Redundancy Check (D10.5)

Correlation heatmap of all divergence and momentum features.
Pairs with |r| > 0.8 are redundancy candidates — one could be
dropped without losing information.

In [ ]:
if features is not None:
    all_cols = sorted(set(div_cols + mom_cols))
    avail = [c for c in all_cols if c in features.columns]

    if len(avail) >= 2:
        corr = features[avail].corr()

        fig, ax = plt.subplots(figsize=(max(8, len(avail) * 0.5), max(6, len(avail) * 0.4)))
        mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
        sns.heatmap(
            corr, mask=mask, annot=len(avail) <= 15, fmt=".2f",
            cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.5, ax=ax,
            xticklabels=True, yticklabels=True,
        )
        ax.tick_params(labelsize=7)
        ax.set_title("Divergence & Momentum Feature Correlation", fontsize=12)
        fig.tight_layout()
        plotting._save_or_show(fig, "12_div_mom_correlation.png", run_cfg)

        # Flag high-correlation pairs
        high_corr = []
        for i in range(len(avail)):
            for j in range(i + 1, len(avail)):
                r = corr.iloc[i, j]
                if abs(r) > 0.8:
                    high_corr.append({
                        "Feature A": avail[i],
                        "Feature B": avail[j],
                        "Correlation": f"{r:.3f}",
                    })

        if high_corr:
            print(f"\nHigh-correlation pairs (|r| > 0.8): {len(high_corr)}")
            display(pd.DataFrame(high_corr))
        else:
            print("\nNo high-correlation pairs (|r| > 0.8) found — features are reasonably independent.")
    else:
        print(f"Only {len(avail)} divergence/momentum columns found — need at least 2 for correlation.")
else:
    print("Features not available.")